# Smart Savings Vault — A/B Test Analysis

Narrative walkthrough of the experiment readout. The heavy lifting lives in
`src/` (so it is tested and reusable); this notebook tells the story.

**Question:** did the Smart Savings Vault feature increase 30-day retention
without hurting transaction quality — and should we ship it?

> Run `python data/generate_data.py && python src/etl.py` first to build the warehouse.

In [1]:
import sys, sqlite3
sys.path.insert(0, "../src")
import pandas as pd
from ab_test import load_user_metrics, run_full_readout, DB_PATH

df = load_user_metrics(DB_PATH)
df.groupby("experiment_group").agg(
    users=("user_id", "size"),
    d30_retention=("retained_d30", "mean"),
    weekly_txns=("weekly_txns", "mean"),
).round(3)

,users,d30_retention,weekly_txns
experiment_group,,,
control,5975,0.435,2.409
treatment,6025,0.491,2.810


## 1. Sanity check — is the randomisation balanced?

Before trusting any lift, confirm the two groups are comparable in size and
composition (a basic *sample ratio mismatch* / balance check).

In [2]:
con = sqlite3.connect(DB_PATH)
users = pd.read_sql_query("SELECT * FROM users", con); con.close()
pd.crosstab(users.plan, users.experiment_group, normalize="columns").round(3)

experiment_group,control,treatment
plan,,
Metal,0.058,0.060
Plus,0.186,0.184
Premium,0.137,0.150
Standard,0.619,0.606


## 2. Primary metric — D30 retention

Two-proportion z-test, with a 95% CI on the absolute lift and a Bayesian
posterior for an intuitive `P(treatment > control)`.

In [3]:
readout = run_full_readout(DB_PATH)
import json
print(json.dumps(readout["primary_metric"], indent=2))
print("Bayesian:", readout["bayesian"])

{
  "name": "d30_retention",
  "control": 43.53,
  "treatment": 49.1,
  "abs_lift": 5.56,
  "rel_lift_pct": 12.78,
  "ci95_low": 3.78,
  "ci95_high": 7.35,
  "p_value": 0.0,
  "significant": true,
  "test": "two-proportion z-test"
}
Bayesian: {'prob_treatment_better': 1.0, 'expected_lift_pp': 5.565, 'cred_interval_95_pp': [3.784, 7.343]}


## 3. Power — could we even detect this?

A minimum-detectable-effect check guards against celebrating an underpowered
result.

In [4]:
readout["power_analysis"]

{'n_per_group': 5975,
 'alpha': 0.05,
 'target_power': 0.8,
 'min_detectable_lift_pp': np.float64(2.549)}

## 4. Secondary & guardrail

Activity should rise; the guardrail (avg transaction value) should not
meaningfully fall.

In [5]:
print(json.dumps(readout["secondary_metric"], indent=2))
print(json.dumps(readout["guardrail_metric"], indent=2))

{
  "name": "weekly_active_transactions",
  "control": 2.409,
  "treatment": 2.81,
  "abs_lift": 0.401,
  "rel_lift_pct": 16.65,
  "ci95_low": 0.333,
  "ci95_high": 0.47,
  "p_value": 0.0,
  "significant": true,
  "test": "Welch t-test (Mann-Whitney p=0.0000)"
}
{
  "name": "avg_transaction_value_guardrail",
  "control": 30.48,
  "treatment": 30.07,
  "abs_lift": -0.41,
  "rel_lift_pct": -1.36,
  "ci95_low": -0.75,
  "ci95_high": -0.07,
  "p_value": 0.01691,
  "significant": true,
  "test": "Welch t-test (guardrail: expect NON-significant)"
}


## 5. Decision

The key judgement: **statistical vs practical significance.** With n ≈ 12k the
guardrail is statistically significant but moves only ~1.4% — below our 2%
practical threshold, so it does not block the launch.

In [6]:
print(readout["decision"])
print(readout["decision_rationale"])

SHIP
Primary metric +5.56pp (p=0.0, 95% CI [3.78, 7.35]). Guardrail moved -1.36% (p=0.01691) - statistically significant but below the 2.0% practical threshold, so not a blocker.


## 6. Does the lift hold within segments?

Aggregate lifts can hide Simpson's-paradox effects. Retention lift should be
positive across plans, not just overall.

In [7]:
seg = df.merge(users[["user_id","plan"]], on="user_id")
(seg.groupby(["plan","experiment_group"])["retained_d30"].mean().mul(100).round(1)
    .unstack().assign(lift_pp=lambda d: (d.treatment-d.control).round(1)))

experiment_group,control,treatment,lift_pp
plan,,,
Metal,52.3,59.6,7.3
Plus,41.5,45.6,4.1
Premium,49.9,57.9,8.0
Standard,41.9,46.9,5.0
